# Daily strategy brief — notebook-generated

Executed by Celery beat (`run_notebook_report`) via papermill inside the
worker container, so `import app.*` and the worker env (DATABASE_URL,
Cloudflare AI, Slack) are all available. Renders the standard
`StrategyReport` plus matplotlib charts embedded as `cid:` images, writes
`report.html` / `report.txt` / chart PNGs and a manifest that the delivery
task emails.

Run it by hand: **Run All** — or via `run_notebook_report("daily_strategy_brief")`.


In [ ]:
# papermill-injected parameters (tagged "parameters")
insight_days = 30
team_id = None            # optional: restrict to one team UUID
output_dir = "/notebooks/output"
run_id = "manual"
manifest_path = ""


In [ ]:
import asyncio
import json
import sys
from pathlib import Path

sys.path.insert(0, "/app")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sqlalchemy import select
from app.models.user import Team
from app.services.strategy_report import build_strategy_report
from app.worker.tasks.digest import _worker_db

OUT = Path(output_dir)
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Build the standard StrategyReport per team — same data assembly, LLM
# playbook, initiatives and pulse as the code path.
async def _build_all():
    reports = []
    async with _worker_db() as db:
        q = select(Team)
        if team_id:
            q = q.where(Team.id == team_id)
        teams = (await db.execute(q)).scalars().all()
        for t in teams:
            reports.append(await build_strategy_report(db, team=t, insight_days=insight_days))
    return reports

reports = await _build_all()
for r in reports:
    print(f"{r.team_name}: {len(r.actions)} actions, "
          f"{len(r.insights.get('platforms') or {})} platforms, "
          f"{len(r.initiatives)} initiatives, llm={r.llm_used}")


In [ ]:
def chart_platform_pulse(report, path):
    """Platform pulse: posts + engagement + avg ER per platform."""
    plats = {
        n: p for n, p in (report.insights.get("platforms") or {}).items()
        if p.get("posts")
    }
    if not plats:
        return False
    names = sorted(plats, key=lambda n: plats[n]["engagement"], reverse=True)
    eng = [plats[n]["engagement"] for n in names]
    er = [plats[n]["avg_engagement_rate"] for n in names]

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.bar(names, eng, color="#0077b5", alpha=0.85, label="engagements")
    ax1.set_ylabel("Engagements (30d)")
    ax1.tick_params(axis="x", rotation=30)
    ax2 = ax1.twinx()
    ax2.plot(names, er, color="#f59e0b", marker="o", label="avg ER %")
    ax2.set_ylabel("Avg engagement rate %")
    ax1.set_title(f"Platform pulse — last {report.insights.get('window_days', 30)}d")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


def chart_initiatives(report, path):
    """Invite funnel + credit balance for LinkedIn page invites."""
    inv = next(
        (i for i in report.initiatives if i["event_type"] == "linkedin_page_invite"),
        None,
    )
    if not inv or not inv.get("units"):
        return False
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.2))
    labels = ["Sent", "Accepted", "Waiting", "Declined"]
    vals = [
        inv["units"],
        inv.get("accepted_est") or 0,
        inv.get("pending_est") or 0,
        inv.get("declined") or 0,
    ]
    ax1.bar(labels, vals, color=["#0077b5", "#10b981", "#f59e0b", "#ef4444"])
    ax1.set_title("Invite funnel")
    cap = inv.get("monthly_cap") or 100
    left = inv.get("credits_left") or 0
    ax2.barh(["credits"], [cap], color="#e5e7eb")
    ax2.barh(["credits"], [left], color="#10b981")
    ax2.set_xlim(0, cap)
    ax2.set_title(f"Credits: ~{left} of {cap} left")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


def chart_engagement_mix(report, path):
    """Likes / comments / shares split per platform."""
    plats = {
        n: p for n, p in (report.insights.get("platforms") or {}).items()
        if p.get("engagement")
    }
    if not plats:
        return False
    names = list(plats)
    likes = [plats[n].get("likes", 0) for n in names]
    comments = [plats[n].get("comments", 0) for n in names]
    shares = [plats[n].get("shares", 0) for n in names]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.bar(names, likes, label="likes", color="#0077b5")
    ax.bar(names, comments, bottom=likes, label="comments", color="#38bdf8")
    ax.bar(names, shares, bottom=[l + c for l, c in zip(likes, comments)],
           label="shares", color="#7dd3fc")
    ax.set_title("Engagement mix (30d)")
    ax.legend()
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


In [ ]:
# Assemble per-team HTML: standard report HTML + charts as cid: images.
# Same skip rule as the code path: no accounts + no impressions = test team.
manifest_reports = []
skipped = []
for idx, rep in enumerate(reports):
    active = (
        rep.digest is not None
        and (
            rep.digest.overview.get("connected_accounts", 0) > 0
            or rep.digest.impressions_24h > 0
        )
    )
    if not active:
        skipped.append(rep.team_name)
        continue
    tag = f"{idx}"
    charts = []
    for name, fn in [
        ("pulse", chart_platform_pulse),
        ("mix", chart_engagement_mix),
        ("initiatives", chart_initiatives),
    ]:
        p = OUT / f"brief-{run_id}-{tag}-{name}.png"
        if fn(rep, p):
            charts.append((name, p))

    charts_html = ""
    if charts:
        imgs = "".join(
            f'<div style="margin:10px 0"><img src="cid:{name}-{tag}" '
            'style="max-width:100%;border-radius:8px"></div>'
            for name, _ in charts
        )
        charts_html = f"<h3>Charts</h3>{imgs}"

    html = rep.to_html()
    if "</body>" in html:
        html = html.replace("</body>", charts_html + "</body>")
    else:
        html += charts_html

    html_file = OUT / f"brief-{run_id}-{tag}.html"
    text_file = OUT / f"brief-{run_id}-{tag}.txt"
    html_file.write_text(html)
    text_file.write_text(rep.to_text())

    manifest_reports.append({
        "subject": rep.subject(),
        "html_file": str(html_file),
        "text_file": str(text_file),
        "attachments": [
            {"file": str(p), "cid": f"{name}-{tag}", "mime": "image/png"}
            for name, p in charts
        ],
    })

if skipped:
    print("skipped empty teams:", ", ".join(skipped))
m_path = Path(manifest_path) if manifest_path else OUT / f"brief-{run_id}.manifest.json"
m_path.write_text(json.dumps({"reports": manifest_reports}, indent=2))
print("manifest →", m_path)
for r in manifest_reports:
    print(" ", r["subject"], "|", len(r["attachments"]), "charts")
